# Notebook 5 — Time-Series Forecasting

## Learning objectives

- Distinguish *direct* and *recursive* forecasting strategies.
- Train one regression model per horizon $h \in \{1, 6, 12\}$ using shifted targets.
- Benchmark every model against the **persistence** baseline $\hat{y}_{t+h} = y_t$.
- Quantify how performance degrades with horizon for different pollutants.
- Forecast AQI categories (classification) instead of values (regression).

## 5.1 What changes when the target is in the future

In Notebooks 2–3 we predicted properties of the *same* hour: $\text{PM}_{2.5}$ at *Aotizhongxin* given $\text{PM}_{2.5}$ at 11 other stations *now*. Forecasting changes the question: predict $\text{PM}_{2.5}$ at *Nongzhanguan* $h$ hours *into the future* given features available *now*.

Two things change. First, the target column becomes a *shifted* version of the original variable:

$$y_{t}^{(h)} = \text{PM}_{2.5,\,t+h}.$$

Second, the train/test split must be chronological (Notebook 4) and the cross-validation procedure must respect time order (Notebook 6).

## 5.2 Direct vs. recursive forecasting

There are two strategies for $h$-step-ahead forecasting.

**Recursive.** Train one 1-step-ahead model. To forecast $h$ steps ahead, call the model $h$ times, feeding each prediction back as the lag-1 feature for the next step. Errors compound.

**Direct.** Train one model *per horizon*. The $h$-step model has target `target.shift(-h)` and is called once. The project uses direct forecasting:

```python
def add_horizon_targets(df, target, horizons):
    for horizon in horizons:
        df[f'target_{slug}_h{horizon}'] = df[target].shift(-horizon)
    return df
```

Default horizons are $h \in \{1, 6, 12\}$ hours. The script generates a target column for each, so we can train three *independent* models on the same feature matrix.

The trade-off: direct forecasting needs more training (one fit per horizon) but produces calibrated multi-horizon predictions and is the strategy of choice in the literature for $h \le 24$ on hourly atmospheric data.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score, f1_score, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_DIR = Path('data')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

csv = DATA_DIR / 'air_quality_nongzhanguan_forecast.csv'
print('Loading:', csv.name)
df = pd.read_csv(csv, parse_dates=['datetime']).sort_values('datetime').reset_index(drop=True)

## 5.3 Linear regression for one-step-ahead

The `baseline_plus_lags` feature set with a `LinearRegression` model and $h = 1$ at *Nongzhanguan* (PM2.5 target) is the headline single-horizon experiment. We will reproduce these numbers below. Reference values from a single offline run:

```
MAE  : 12.90
MAPE : 42.93%
RMSE : 22.02
R2   : 0.940
```

with 26,456 training rows and 6,614 test rows. The 50 features are 33 lag columns, 6 cyclic columns, 10 current-hour numeric columns, and 1 categorical column expanded by one-hot to 16 indicator columns. **The dominant predictor is `PM2.5_lag_1`**: the next hour's $\text{PM}_{2.5}$ is, on average, very close to the current hour's. We will reproduce these numbers below.

In [ ]:
horizons = [1, 6, 12]
lag_cols = [c for c in df.columns if '_lag_' in c]
CYCLIC = ['hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos']
METEO  = ['TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']
CURRENT_NUMERIC = ['PM10', 'SO2', 'NO2', 'CO', 'O3'] + METEO
NUMERIC = CURRENT_NUMERIC + CYCLIC + lag_cols

cut = int(0.8 * len(df))
train_df, test_df = df.iloc[:cut], df.iloc[cut:]
print(f'Train rows: {len(train_df):,}  Test rows: {len(test_df):,}')
print(f'Numeric features: {len(NUMERIC)}  +  1 categorical (wd, one-hot)')

### The persistence baseline

Before evaluating any ML model, we benchmark the **persistence** forecast:

$$\hat{y}_{t+h}^{\text{persist}} = y_{t}.$$

This is what you would predict if you had no model at all — "tomorrow looks like today". For autoregressive pollutants persistence is shockingly competitive at $h=1$. Any ML model worth deploying must beat persistence at *every* horizon.

In [ ]:
def fit_predict_h(horizon):
    target = f'target_pm25_h{horizon}'
    pre = ColumnTransformer([
        ('num', StandardScaler(), NUMERIC),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['wd']),
    ])
    pipe = Pipeline([('pre', pre), ('lr', LinearRegression())])
    pipe.fit(train_df[NUMERIC + ['wd']], train_df[target])
    pred = pipe.predict(test_df[NUMERIC + ['wd']])
    mae  = mean_absolute_error(test_df[target], pred)
    r2   = r2_score(test_df[target], pred)
    # Persistence baseline: predict y_t (the last observed PM2.5).
    persist = test_df['PM2.5'].values
    mae_p   = mean_absolute_error(test_df[target], persist)
    r2_p    = r2_score(test_df[target], persist)
    return {'h': horizon, 'MAE_lr': mae, 'R2_lr': r2,
            'MAE_persistence': mae_p, 'R2_persistence': r2_p}

rows = [fit_predict_h(h) for h in horizons]
result = pd.DataFrame(rows).set_index('h')
result

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
result[['MAE_lr', 'MAE_persistence']].plot(kind='bar', ax=axes[0], color=['steelblue', 'lightgray'])
axes[0].set_title('MAE vs horizon (lower is better)'); axes[0].set_ylabel('μg/m³')
result[['R2_lr', 'R2_persistence']].plot(kind='bar', ax=axes[1], color=['seagreen', 'lightgray'])
axes[1].set_title('R² vs horizon (higher is better)')
for ax in axes:
    ax.set_xlabel('horizon (hours)'); ax.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

**Interpretation.** At $h=1$ the model barely beats persistence — $\text{PM}_{2.5}$ is highly autoregressive, so simply repeating the last value is a strong baseline. At $h=12$ persistence collapses (a 12-hour-old reading is no longer informative) but the regression keeps explaining a substantial share of variance via meteorology and the diurnal cycle.

## 5.4 Linear regression for 12-hour-ahead

Repeating the same experiment at horizon $h = 12$ shows substantial degradation: $R^{2}$ falls from 0.94 to roughly 0.5, and MAE roughly doubles. **Why?** At $h = 1$ the autoregressive term `PM2.5_lag_1` carries almost the entire signal. At $h = 12$, the half-day weather state has shifted, the diurnal cycle has gone through its full descent, and persistence alone is no longer informative. The model now has to lean on the cyclic features and the meteorology rather than on lags. Linear regression is not flexible enough to express the resulting non-linear relationship between weather and pollutant; gradient boosting (Notebook 7) closes most but not all of the gap.

### Forecast trace for h = 1

A quick visual check: predicted vs actual over the first 500 hours of the test set.

In [ ]:
target = 'target_pm25_h1'
pre = ColumnTransformer([('num', StandardScaler(), NUMERIC),
                          ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['wd'])])
pipe = Pipeline([('pre', pre), ('lr', LinearRegression())])
pipe.fit(train_df[NUMERIC + ['wd']], train_df[target])
pred1 = pipe.predict(test_df[NUMERIC + ['wd']])

N = 500
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(test_df['datetime'].iloc[:N], test_df[target].iloc[:N], label='Actual', lw=1.5)
ax.plot(test_df['datetime'].iloc[:N], pred1[:N], label='Predicted', lw=1.5, alpha=0.85)
ax.set_title('1-hour-ahead PM2.5 forecast (first 500 test hours)')
ax.set_ylabel('PM2.5 (μg/m³)'); ax.legend()
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30); plt.tight_layout(); plt.show()

## 5.5 Logistic regression for AQI prediction

The same forecasting framework supports classification: replace the continuous shifted target by a *categorical* shifted target. For the 3-class scheme of Notebook 3:

$$y_{t}^{(h)} = \begin{cases} \text{Low} & \text{if } \text{PM}_{2.5,\,t+h} \le 35,\\ \text{Medium} & \text{if } 35 < \text{PM}_{2.5,\,t+h} \le 75, \\ \text{High} & \text{if } \text{PM}_{2.5,\,t+h} > 75. \end{cases}$$

Train a multinomial logistic regression on the same `baseline_plus_lags` features and evaluate with macro-$F_{1}$, accuracy and a 3 × 3 confusion matrix.

For $h = 1$, the macro-$F_{1}$ should approach 0.85+ (the regression's $R^{2}$ near 0.94 implies very few category boundary crossings will be missed). For $h = 12$, it falls into the 0.55–0.65 range, dominated by misclassifications of *Medium* hours that drift across the 35 or 75 boundary.

In [ ]:
def aqi3(s):
    return pd.cut(s, bins=[-np.inf, 35, 75, np.inf], labels=['Low', 'Medium', 'High'])

y_h1_class_train = aqi3(train_df[target])
y_h1_class_test  = aqi3(test_df[target])
mask_tr = y_h1_class_train.notna(); mask_te = y_h1_class_test.notna()

pipe_clf = Pipeline([
    ('pre', ColumnTransformer([
        ('num', StandardScaler(), NUMERIC),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['wd'])])),
    ('clf', LogisticRegression(max_iter=1000)),
])
pipe_clf.fit(train_df.loc[mask_tr, NUMERIC + ['wd']], y_h1_class_train[mask_tr])
pred_class = pipe_clf.predict(test_df.loc[mask_te, NUMERIC + ['wd']])

print(f"Accuracy : {accuracy_score(y_h1_class_test[mask_te], pred_class):.3f}")
print(f"Macro F1 : {f1_score(y_h1_class_test[mask_te], pred_class, average='macro'):.3f}")

## 5.6 Backtesting and rolling-window evaluation

A single chronological train/test split estimates one number: out-of-sample performance during one period. Because the test period is fixed (the last 20% of data), results are sensitive to whatever happened then. **Backtesting** with a rolling origin generates a sequence of evaluations at successive cut-off times $t_{\text{cut}}^{(1)} < t_{\text{cut}}^{(2)} < \dots$, and reports the distribution of metrics. `sklearn.model_selection.TimeSeriesSplit(n_splits=5)` does exactly this — we use it inside `GridSearchCV` in Notebook 6:

```
[train: 1...10][test: 11...12]
[train: 1...12][test: 13...14]
[train: 1...14][test: 15...16]
[train: 1...16][test: 17...18]
[train: 1...18][test: 19...20]
```

## 5.7 Reproducing a multi-horizon, multi-feature-set comparison

The cell below runs the same kind of sweep that is described in the literature for this data: every combination of `{horizon} × {feature_set} × {model}` is fitted and scored on the held-out test set. We restrict the model family to two (LinearRegression and GradientBoostingRegressor) and the pollutant to two ($PM_{2.5}$ and $O_3$ — the two forecasting CSVs that ship in `data/`). That is $2 \times 3 \times 3 \times 2 = 36$ experiments, which takes roughly one minute on a modern laptop.

From the resulting summary table you can verify the following empirical observations:

1. **No universally best model.** GBM dominates $O_3$ (typically $-20\%$ MAE or more vs linear). Linear is hard to beat for $PM_{2.5}$ at $h=1$.
2. **Lags beat current-hour features** for autoregressive pollutants — compare `lags_only` and `baseline` across the table.
3. **Degradation with horizon depends on the pollutant.** Look at $PM_{2.5}$ R² from $h=1$ to $h=12$ vs the same for $O_3$.
4. **`baseline_plus_lags` with GBM is typically the strongest configuration.** It wins the majority of the (pollutant × horizon) cells.

Add more pollutant CSVs to `data/` (generated from the raw `PRSA_Data_*.csv` files) to extend the sweep to the full six-pollutant comparison.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

POLLUTANT_FILES = {
    'PM2.5': DATA_DIR / 'air_quality_nongzhanguan_forecast.csv',
    'O3'   : DATA_DIR / 'air_quality_nongzhanguan_o3_forecast.csv',
}
TARGET_SLUG = {'PM2.5': 'pm25', 'O3': 'o3'}
HORIZONS    = [1, 6, 12]
FEATURE_SETS = ['baseline', 'lags_only', 'baseline_plus_lags']
MODELS = {
    'linear': lambda: LinearRegression(),
    'gbm'   : lambda: GradientBoostingRegressor(
                  n_estimators=200, max_depth=3, learning_rate=0.1, random_state=0),
}

def build_feature_lists(df_, pollutant, feature_set):
    lag_cols_ = [c for c in df_.columns if '_lag_' in c]
    cyclic = ['hour_sin','hour_cos','month_sin','month_cos','day_of_week_sin','day_of_week_cos']
    meteo  = ['TEMP','PRES','DEWP','RAIN','WSPM']
    other_pollutants = [p for p in ['PM2.5','PM10','SO2','NO2','CO','O3'] if p != pollutant]
    current_numeric = [c for c in other_pollutants + meteo if c in df_.columns]
    if feature_set == 'baseline':
        return current_numeric + cyclic
    if feature_set == 'lags_only':
        return lag_cols_ + cyclic
    return current_numeric + cyclic + lag_cols_  # baseline_plus_lags

rows = []
for pollutant, path in POLLUTANT_FILES.items():
    df_p = pd.read_csv(path, parse_dates=['datetime']).sort_values('datetime').reset_index(drop=True)
    cut_p = int(0.8 * len(df_p))
    train_p, test_p = df_p.iloc[:cut_p], df_p.iloc[cut_p:]
    for h in HORIZONS:
        target_col = f'target_{TARGET_SLUG[pollutant]}_h{h}'
        for fset in FEATURE_SETS:
            feats = build_feature_lists(df_p, pollutant, fset)
            for mname, mfactory in MODELS.items():
                pre_ = ColumnTransformer([
                    ('num', StandardScaler(), feats),
                    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['wd']),
                ])
                pipe_ = Pipeline([('pre', pre_), ('model', mfactory())])
                pipe_.fit(train_p[feats + ['wd']], train_p[target_col])
                pred_ = pipe_.predict(test_p[feats + ['wd']])
                rows.append({
                    'pollutant': pollutant, 'h': h,
                    'feature_set': fset, 'model': mname,
                    'MAE': mean_absolute_error(test_p[target_col], pred_),
                    'R2' : r2_score(test_p[target_col], pred_),
                })

sweep = pd.DataFrame(rows)
print(sweep.pivot_table(index=['pollutant','feature_set'], columns=['h','model'], values='MAE').round(2))

In [ ]:
# Visualise: for each pollutant, show how MAE depends on (feature_set, model) at each horizon.
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)
for ax, pollutant in zip(axes, ['PM2.5', 'O3']):
    sub = sweep[sweep['pollutant'] == pollutant]
    pivot = sub.pivot_table(index='h', columns=['feature_set', 'model'], values='MAE')
    pivot.plot(kind='bar', ax=ax)
    ax.set_title(f'{pollutant} — MAE by horizon × (feature_set, model)')
    ax.set_ylabel('MAE'); ax.set_xlabel('horizon (h)')
    ax.legend(fontsize=8, ncol=2)
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

## 5.8 Common misconceptions

- **"Recursive forecasting is more elegant."** It is more elegant in *theory* (one model fits all horizons) but suffers in *practice* because errors compound multiplicatively. Direct forecasting wins on hourly atmospheric data for every horizon $\le 24$.
- **"A 24-hour forecast is just a 1-hour forecast made 24 times."** Only if you accept compounded error. With direct forecasting, $\text{PM}_{2.5}$ at $t+24$ is predicted *directly* from features at $t$.
- **"$R^{2}$ near 1 at $h=1$ proves my model is good."** It mostly proves $\text{PM}_{2.5}$ is autocorrelated. The lag-1 *persistence* baseline already achieves $R^{2}$ around 0.93. Always benchmark against persistence.
- **"Random KFold should still be OK if my data is large."** It is not OK at any scale: it leaks future into past. Use `TimeSeriesSplit`.

---
## Exercises

### Exercise 5.1 — sweep horizons 1, 3, 6, 9, 12

*Hint: the forecasting CSV in `data/` contains three pre-built shifted targets — `target_pm25_h1`, `target_pm25_h6`, `target_pm25_h12`. Loop over these three horizons, fit one regression per horizon, and plot MAE as a function of $h$. Add a second line for the persistence baseline so you can see the gap close at long horizons.*

In [ ]:
# EXERCISE 5.1
# TODO: produce a line plot of MAE vs horizon, with a separate line for the persistence baseline.


### Exercise 5.2 — recursive forecasting demo

*Hint: train a single 1-step-ahead model, then iterate it 12 times feeding each prediction back as `PM2.5_lag_1`. Compare MAE to the direct $h=12$ model. Why does recursive lose? (Errors at each step add to the input of the next step.)*

In [ ]:
# EXERCISE 5.2
# TODO: implement a 12-step recursive forecast for the first 100 test rows and compare.


### Exercise 5.3 — peak-hour analysis

*Hint: compute MAE only on hours where the actual PM2.5 exceeds 150 μg/m³ (heavy-pollution episodes). Is the model worse on those hours? Why does this matter for public-health alerts?*

In [ ]:
# EXERCISE 5.3
# heavy_mask = test_df[target] > 150
# TODO: report MAE on heavy_mask vs ~heavy_mask.


## 5.9 Chapter summary

- Forecasting reframes the target as a future-shifted version of the variable; everything else (features, model, loss) carries over from spatial regression.
- Direct forecasting (one model per horizon, target `shift(-h)`) avoids the compounding errors of recursive forecasting and is our default.
- Performance degrades with horizon because the autoregressive lag-1 signal weakens; the rate of degradation is pollutant-specific (slowest for O$_{3}$, fastest for SO$_{2}$).
- The same direct-forecasting framework supports classification of future AQI categories simply by replacing the continuous shifted target with a categorical one.
- `TimeSeriesSplit` is mandatory for cross-validation on time-ordered data; never use `KFold` or `train_test_split(shuffle=True)`.

**Next:** Notebook 6 introduces non-linear models (decision trees and random forests).